In [0]:
table_name = "dim_date"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_date")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import col, to_date, upper, trim, when

# 1. Safe Cast to DATE (TRY_TO_DATE equivalent in Spark: to_date returns null if it fails)
df = df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

# 2. Standardize text: UPPER(TRIM())
df = df.withColumn("day_of_week", upper(trim(col("day_of_week"))))

# 3. Data Type Optimization: TRY_CAST to INT (Spark cast turns invalid to null)
df = df.withColumn("quarter", trim(col("quarter")).cast("int"))

# 4. Cast to INT & fix future years: If year > 2025 -> NULL
df = df.withColumn("year", when(trim(col("year")).cast("int") > 2025, None)
                               .otherwise(trim(col("year")).cast("int")))

# 5. Boolean Optimization: Map 'yes'/'1' -> TRUE, 'no'/'0' -> FALSE
df = df.withColumn("is_holiday", 
                   when(upper(trim(col("is_holiday"))).isin("YES", "1"), True)
                   .when(upper(trim(col("is_holiday"))).isin("NO", "0"), False)
                   .otherwise(None).cast("boolean"))

# Drop audit columns (they are not needed in Silver onward per standard mapping)
df = df.drop("_ingested_at", "_source_file")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")